# Twisting brackets, second pass: the same question on the research interface

This is the companion of `bracket_twist_walkthrough.ipynb`. That notebook
solved the question of [arXiv:2409.11973, §7] — *twist the Dorfman bracket by
`Ψ = (1 Π; 0 1)` and recover the Nambu–Poisson tilde calculus; then rotate the
exceptional Courant bracket by the E₆ matrix `Ψ_Π` and check everything* — with
the tools the code base had, and it ended with a list of five things a user
had to do by hand: write matrices as functions, carry sections as bare tuples,
assemble the rewrite engine rule by rule, write each condition check
individually, and antisymmetrise indices manually.

Those five things now exist as a small layer, `jacopy.research`:

| brick | what it gives you |
| --- | --- |
| `SectionType`, `GeneralizedSection` | typed sections `U ⊕ ω₂ ⊕ ω₅` that add, scale and know their slots |
| `BlockMatrix`, `Map` | `Ψ` written as a matrix; action, products and the inverse are *derived* |
| `assemble_engine` | the rewrite engine built from the expressions you check, with a report of what it added |
| `AlgebroidData`, `AxiomSuite` | the bracket with its anchor / pairing / `D` / R-valued action; one call runs the Courant-type conditions it knows (right-Leibniz, symmetric part, anchor morphism, metric invariance, Leibniz–Jacobi — *not* the Appendix-D calculus/compatibility families, which stay separate provers); `.transport(Ψ)` is eq. (7.3) for the whole structure |
| `antisymmetrize` | `T^{[a₁…a_n]}` on frame-component expressions |

The mathematics is unchanged; what changes is how much a reader has to type
and how much the output explains itself. As before, `CLOSED` means the engine
reduced a difference to literal `0`, `RESIDUAL` means it did not and shows
what is left, and nothing is assumed silently — the Poisson condition is an
explicit flag wherever it is used.

In [ ]:
import time
from jacopy.core.registry import PropertyRegistry
from jacopy.core.expr import Integer, Neg, Sum
from jacopy.core.wedge import Wedge
from jacopy.algebra.derivation import Act
from jacopy.central.objects import forms, functions, vector_fields
from jacopy.central.tangent.cartan import L as Lie
from jacopy.central.tangent.exterior import d
from jacopy.packages.poisson.nambu import nambu_structure
from jacopy.packages.poisson.tilde import _normalized_by      # engine normal form of an expression
from jacopy.research import (
    SectionType, BlockMatrix, Map, Bracket, AlgebroidData, AxiomSuite,
    assemble_engine, assembly_report, antisymmetrize, swap_indices,
)

reg = PropertyRegistry()
f, g, h = functions("f g h", registry=reg)     # h is the probe for vector-valued identities
U, V, W = vector_fields("U V W")

## Part 1 — Dorfman bracket twisted by `Ψ = (1 Π; 0 1)`

### 1.1 Sections and the matrix, as objects

`T = TM ⊕ T*M` is a *type*; `T.section(U, ω)` is a section. The bivector `θ`
enters as the operator `θ♯ : T*M → TM` (`Map(N.sharp_vf, "θ♯")`), and the
matrix is written as a matrix. Its inverse is computed — `Ψ` is unipotent, so
`Ψ⁻¹ = 1 − N` with `N = Ψ − 1` — and we let the engine confirm `Ψ⁻¹Ψ = id`.

In [ ]:
T = SectionType.generalized_tangent()          # TM ⊕ Λ¹T*M
om, et, mu = forms("ω η μ", degree=1)
N = nambu_structure("θ", p=1)                  # the bivector θ

e1, e2, e3 = T.section(U, om), T.section(V, et), T.section(W, mu)
Psi = BlockMatrix(T, [[1, Map(N.sharp_vf, "θ♯")], [0, 1]], name="Ψ_θ")
print(Psi)
print(Psi.inverse())
print("\nΨ_θ e₁      =", Psi(e1))
back = Psi.inverse()(Psi(e1))
print("Ψ_θ⁻¹Ψ_θ e₁ =", back, "  → normalised difference from e₁:",
      [str(_normalized_by(assemble_engine(c, registry=reg, structures=(N,)), Sum(c, Neg(o)), reg)) for c, o in zip(back, e1)])

### 1.2 The Dorfman structure and its axiom suite

An `AlgebroidData` bundles the bracket with its anchor `ρ(U ⊕ ω) = U`, the
canonical pairing `⟨e₁,e₂⟩ = ι_U η + ι_V ω`, the coboundary
`D⟨e₁,e₂⟩ = 0 ⊕ d⟨e₁,e₂⟩` and the action `ℒ^R` of a section on the
pairing's values (here a scalar, so `ℒ^R_e g = ρ(e)(g)`). `AxiomSuite.run`
checks right-Leibniz, the symmetric part and the anchor-morphism condition,
component by component — and, given a third section, the metric-invariance
condition `ℒ^R_{e₁}⟨e₂,e₃⟩ = ⟨[e₁,e₂],e₃⟩ + ⟨e₂,[e₁,e₃]⟩` and the
Leibniz–Jacobi identity. Those are the Courant-type conditions the suite
knows; the calculus/compatibility families of Appendix D remain the separate
provers of the first notebook.

In [ ]:
from jacopy.packages.drinfeld.double import dorfman_double, canonical_pairing

standard = AlgebroidData(
    bracket=Bracket.from_components(T, dorfman_double, name="Dorfman"),
    anchor=lambda e: e[0],
    pairing=lambda a, b: (canonical_pairing(*a, *b),),
    D=lambda p: T.section(Integer(0), d(p)),
    lie_R=lambda e, g: (Act(e[0], g),),          # ℒ^R on the scalar pairing: the anchor acting
    name="Dorfman on TM⊕T*M",
)
suite = AxiomSuite(standard, registry=reg, probe=h, structures=(N,))
print(suite.run(e1, e2, f, e3=e3))

### 1.3 The twist — one line

`standard.transport(Psi)` is eq. (7.3) applied to the *whole* structure: the
bracket becomes `Ψ⁻¹[Ψ·,Ψ·]`, the anchor `ρ∘Ψ`, the pairing `⟨Ψ·,Ψ·⟩` and the
coboundary `Ψ⁻¹∘D`. The suite runs on the twisted structure unchanged. (The
Leibniz–Jacobi identity of the twisted bracket closes too — 3600-odd steps
per component, about a minute each — so it lives in the slow test suite
rather than in this notebook.)

Notice what the suite does **not** need: any assumption on `θ`. The twisted
structure is isomorphic to the Dorfman one through `Ψ`, so every axiom
transports for an arbitrary bivector. The Poisson condition only enters in
the *identification* of the twisted bracket with the Nambu double — its
vector part differs by the derived twist `R′(ω,η) = [θ♯ω, θ♯η] − θ♯[ω,η]_θ`,
which vanishes exactly when `θ` is Poisson. We show both facts.

In [ ]:
from jacopy.packages.drinfeld.double import nambu_double
from jacopy.packages.drinfeld.twist import r_twist

twisted = standard.transport(Psi)
print("twisted bracket [e₁,e₂]_Ψ =", twisted.bracket(e1, e2), "\n")
twisted_suite = AxiomSuite(twisted, registry=reg, probe=h, structures=(N,))
print(twisted_suite.run(e1, e2, f))

# identification with the Nambu double (the tilde calculus of Part 1 of the first notebook)
tw = twisted.bracket(e1, e2)
nb = T.section(*nambu_double(N, U, om, V, et))
eng = assemble_engine(*tw, *nb, registry=reg, structures=(N,))
print("\nform part:   [e₁,e₂]_Ψ − Nambu double        =", _normalized_by(eng, Sum(tw[1], Neg(nb[1])), reg))
print("vector part: [e₁,e₂]_Ψ − Nambu double − R′(ω,η) =",
      _normalized_by(eng, Sum(tw[0], Neg(nb[0]), Neg(r_twist(N, om, et))), reg))
eng_fi = assemble_engine(r_twist(N, om, et), registry=reg, structures=(N,), declare_fi=True)
print("R′(ω,η) under the DECLARED Poisson condition   =", _normalized_by(eng_fi, r_twist(N, om, et), reg))
print("R′(ω,η) without it                             =", _normalized_by(eng, r_twist(N, om, et), reg)._repr_inner()[:80], "…")

### 1.4 The engine explains itself

Every check above built its own engine from the expressions involved. The
report lists what was detected and which rules were added because of it —
this is the knowledge a user of the first notebook had to bring by hand.

In [ ]:
print("\n".join(twisted_suite.last_report))

The calculus conditions (3 + 2), the Appendix-D compatibility conditions
(D.5)–(D.19) and the bracket-morphism conditions of the tilde calculus are
*theorems about the identified structure*; they were run one by one in the
first notebook and are unchanged. Here we re-run the three that need the
Poisson declaration, to show the flag is the same explicit `declare_fi`.

In [ ]:
from jacopy.packages.drinfeld.tilde_calculus import (
    prove_tilde_calculus_condition_one, prove_tilde_calculus_condition_two, prove_tilde_calculus_condition_three,
)
for label, prover in (("(D.5)", prove_tilde_calculus_condition_one),
                      ("(D.6)", prove_tilde_calculus_condition_two),
                      ("(D.7)", prove_tilde_calculus_condition_three)):
    chain, _ = prover(N, om, et, W, f, h, registry=reg, declare_fi=True)
    print(f"CLOSED  {label} tilde calculus condition ({len(chain.steps)} steps, declared Poisson condition)")

### 1.5 The same at higher order: a trivector, a 4-vector

Nothing above was special to a bivector. Replace `θ` by a `(p+1)`-vector
`Π` acting as `Λᵖ → TM`, the bundle by `TM ⊕ ΛᵖT*M`, and the same matrix
`Ψ_Π = (1 Π; 0 1)` twists the Dorfman bracket into the Nambu–Poisson tilde
calculus of order `p` (the higher Koszul bracket `ℒ_{Πω}η − ι_{Πη}dω`). The
suite, the transport, the identification with the Nambu double up to `R′`,
and the Appendix-D conditions (D.5)–(D.14) and (D.16)–(D.19) run unchanged
((D.15) is the definition of the action, not a condition) — the only thing a user has to
supply differently is the R-valued action on the pairing, which for `p ≥ 2`
is a `(p−1)`-form, so `ℒ^R` is the Lie derivative `ℒ_U` rather than the
scalar action `U(·)`. Here is `p = 2` in full (the Leibniz–Jacobi identity
of the base Dorfman bracket on `Λ²` takes ~15 s; at `p = 3` it takes ~100 s
and is left to the test suite, everything else at `p = 3` is seconds).

In [ ]:
from jacopy.central.tangent.cartan import L as Lie
import jacopy.packages.drinfeld.tilde_calculus as tc
import jacopy.packages.drinfeld.metric_invariance as mi
import jacopy.packages.drinfeld.bracket_morphism as bm

def part_one_at_order(p, *, jacobi):
    t0 = time.time()
    Np = nambu_structure("Π", p=p)
    Tp = SectionType.generalized_tangent(p)
    a, b, c = forms("ω η μ", degree=p)
    slots_m1 = tuple(vector_fields(" ".join(f"Y{i}" for i in range(1, p))))     # a (p−1)-form is checked on p−1 vectors
    slots_p = tuple(vector_fields(" ".join(f"Y{i}" for i in range(1, p + 1))))   # a p-form on p vectors
    s1, s2, s3 = Tp.section(U, a), Tp.section(V, b), Tp.section(W, c)
    Psi_p = BlockMatrix(Tp, [[1, Map(Np.sharp_vf, "Π")], [0, 1]], name="Ψ_Π")
    base = AlgebroidData(
        bracket=Bracket.from_components(Tp, dorfman_double, name="Dorfman"),
        anchor=lambda e: e[0], pairing=lambda x, y: (canonical_pairing(*x, *y),),
        D=lambda q: Tp.section(Integer(0), d(q)),
        lie_R=lambda e, g: (Lie(e[0], g),),                       # ℒ_U on the (p−1)-form pairing
        name=f"Dorfman on TM⊕Λ{p}")
    rep = AxiomSuite(base, registry=reg, probe=h, structures=(Np,)).run(s1, s2, f, e3=s3 if jacobi else None)
    rep.extend(AxiomSuite(base.transport(Psi_p), registry=reg, probe=h, structures=(Np,)).run(s1, s2, f))
    print(rep)
    tw = base.transport(Psi_p).bracket(s1, s2); nb_ = Tp.section(*nambu_double(Np, U, a, V, b))
    e_ = assemble_engine(*tw, *nb_, registry=reg, structures=(Np,))
    print("identification: form part − Nambu double =", _normalized_by(e_, Sum(tw[1], Neg(nb_[1])), reg),
          "| vector part − Nambu double − R′ =", _normalized_by(e_, Act(Sum(tw[0], Neg(nb_[0]), Neg(r_twist(Np, a, b))), h), reg))
    checks = [
        ("(D.5)", lambda: tc.prove_tilde_calculus_condition_one(Np, a, b, W, f, h, registry=reg)),
        ("(D.6)", lambda: tc.prove_tilde_calculus_condition_two(Np, a, b, W, f, h, registry=reg)),
        ("(D.7)", lambda: tc.prove_tilde_calculus_condition_three(Np, a, b, W, f, h, registry=reg)),
        ("(D.8)", lambda: tc.prove_jacobi_compat_d8(Np, U, b, c, registry=reg)),
        ("(D.9)", lambda: tc.prove_jacobi_compat_d9(Np, U, b, c, slots_p, registry=reg)),
        ("(D.10)", lambda: tc.prove_jacobi_compat_d10(Np, a, b, W, registry=reg)),
        ("(D.11)", lambda: tc.prove_jacobi_compat_d11(Np, a, V, W, f, h, registry=reg)),
        ("(D.12)", lambda: tc.prove_jacobi_compat_d12(Np, a, V, W, h, registry=reg)),
        ("(D.13)", lambda: tc.prove_jacobi_compat_d13(Np, c, U, V, h, registry=reg)),
        ("(D.14)", lambda: mi.prove_z_metric_invariance(Np, a, b, c, slots_m1, registry=reg)),
        ("(D.16)", lambda: mi.prove_a_mixing_condition(Np, U, V, c, slots_m1, registry=reg)),
        ("(D.17)", lambda: mi.prove_a_invariance_of_gz(Np, U, b, c, slots_m1, registry=reg)),
        ("(D.18)", lambda: mi.prove_dual_mixing_condition(Np, a, b, W, slots_m1, registry=reg)),
        ("(D.19)", lambda: mi.prove_double_metric_invariance(Np, U, a, V, b, W, c, slots_m1, registry=reg)),
        ("(4.19) total anchor is a bracket morphism", lambda: bm.prove_total_anchor_is_bracket_morphism(Np, U, a, V, b, h, registry=reg)),
    ]
    line = []
    for label, thunk in checks:
        out = thunk(); chain = out[0] if isinstance(out, tuple) else out
        line.append(f"{label} {len(chain.steps)}")
    print("CLOSED (steps):", ", ".join(line), f"  [{time.time()-t0:.1f}s total]")

part_one_at_order(2, jacobi=True)
print()
part_one_at_order(3, jacobi=False)

### 1.6 Two dialects, one bracket

The code base grew two node families for the same order-1 objects: the
Poisson package writes `θ♯ω` and the Koszul bracket as an atom
`[ω,η]_π = ℒ_{π♯ω}η − ℒ_{π♯η}ω − dπ(ω,η)`; the Nambu package (used
throughout this notebook) writes `Π(ω)` and the formula `ℒ_{Πω}η − ι_{Πη}dω`.
The engine never identified them, so the Jacobi identity of the Koszul
bracket — already a library theorem in the Poisson dialect — had to be
re-proved in the Nambu dialect when the Watamura [C′1] identity was closed.
A single definitional rule, `Π(ω) → π♯(ω)` on the same bivector, bridges
the dialects: the two Koszul brackets agree for *any* bivector
(declaration-free), and the form part of [C′1] then closes by *citing* the
Poisson-dialect theorem (about 45 s, almost all of it the cited proof).

In [ ]:
from jacopy.packages.generalized.dialect_bridge import prove_koszul_dialects_agree, prove_theta_form_jacobi_by_citation

(X,) = vector_fields("X")                     # a probe vector: 1-form identities are checked on it
t = time.time()
chain, thm = prove_koszul_dialects_agree(N, om, et, X, registry=reg)
print(f"CLOSED  {thm.statement[:78]}…  ({time.time()-t:.2f}s, {thm.notes})")
t = time.time()
chain, thm = prove_theta_form_jacobi_by_citation(N, om, et, mu, X, f, registry=reg)
for s in chain.steps:
    print(f"CLOSED  {s.rule[:96]}" + (f"  [{len(s.children)} cited sub-steps]" if s.children else ""))
print("assumptions:", thm.from_axioms, f"({time.time()-t:.1f}s)")

## Part 2 — the exceptional Courant bracket rotated by `Ψ_Π`

### 2.1 Type, bracket and the E₆ matrix

`T₃ = TM ⊕ Λ²T*M ⊕ Λ⁵T*M`. The exceptional bracket is the library formula
(Dorfman on both form slots plus the cross-term `−η₂ ∧ dω₂`), the two
pairings are a 1-form and a 4-form, and `D` maps them to `0 ⊕ dP₂ ⊕ dP₅`.
The rotation (4.10)–(4.14) is a genuine 3×3 block matrix now: entries
`Π₃`, `Π₆` and the ⊛ map, with `Π₃ ⊛ Π₃` entered as the *sum* of two
operators in one cell. The inverse is again derived.

In [ ]:
from jacopy.packages.drinfeld.examples import (
    exceptional_courant_bracket, exceptional_pairing_two, exceptional_pairing_five, boxtimes,
)

T3 = SectionType.exceptional()
N3 = nambu_structure("Π₃", p=2)      # trivector   Π₃ : Λ² → TM
N6 = nambu_structure("Π₆", p=5)      # 6-vector    Π₆ : Λ⁵ → TM
om2, et2, ze2 = forms("ω₂ η₂ ζ₂", degree=2)
om5, et5, ze5 = forms("ω₅ η₅ ζ₅", degree=5)
x1, x2, x3 = T3.section(U, om2, om5), T3.section(V, et2, et5), T3.section(W, ze2, ze5)

exceptional = AlgebroidData(
    bracket=Bracket.from_components(T3, exceptional_courant_bracket, name="exceptional Courant"),
    anchor=lambda e: e[0],
    pairing=lambda a, b: (exceptional_pairing_two(a[0], a[1], b[0], b[1]),
                          exceptional_pairing_five(*a, *b)),
    D=lambda p2, p5: T3.section(Integer(0), d(p2), d(p5)),
    # the R-VALUED action on the pairing values (r₁, r₄): NOT ℒ_U on both —
    # the 4-form component picks up −r₁ ∧ dω₂ from the cross-term
    lie_R=lambda e, r1, r4: (Lie(e[0], r1), Sum(Lie(e[0], r4), Neg(Wedge(r1, d(e[1]))))),
    name="exceptional on TM⊕Λ²⊕Λ⁵",
)
print("[x₁,x₂] =", exceptional.bracket(x1, x2), "\n")

P3 = Map(N3.sharp_vf, "Π₃")
P6 = Map(N6.sharp_vf, "Π₆")
BX = Map(lambda w5: boxtimes(N3, w5), "Π₃⊛Π₃")          # (Π₃⊛Π₃)(ω₅) = ½ Π₃(ι_{Π₃} ω₅)
PsiPi = BlockMatrix(T3, [[1, P3, P6 + BX],
                         [0, 1, 0],
                         [0, 0, 1]], name="Ψ_Π")
print(PsiPi)
print(PsiPi.inverse())
print("\nΨ_Π x₁ =", PsiPi(x1))

### 2.2 Conditions of the unrotated and of the rotated bracket

First the exceptional bracket itself (right-Leibniz, symmetric part, anchor
morphism, metric invariance with the R-valued action `ℒ^R_{(U,ω₂,ω₅)}(r₁,r₄)
= (ℒ_U r₁, ℒ_U r₄ − r₁ ∧ dω₂)` — the plain `ℒ_U` on both components would
be wrong — and the Leibniz–Jacobi identity on the vector and 2-form slots;
the 5-form slot with the cross-term is discussed at the end). Then
`exceptional.transport(PsiPi)` and the same suite on the rotated structure,
with the transported anchor `ρ'(x) = U + Π₃ω₂ + Π₆ω₅ + (Π₃⊛Π₃)ω₅` and the
transported pairings.

In [ ]:
t = time.time()
base_suite = AxiomSuite(exceptional, registry=reg, probe=h, structures=(N3, N6))
report = base_suite.run(x1, x2, f)
report.extend(base_suite.metric_invariance(x1, x2, x3))
report.extend(base_suite.jacobi(x1, x2, x3, components=(0, 1), max_steps=4000))
print(report, f"\n({time.time()-t:.1f}s)")

In [ ]:
t = time.time()
rotated = exceptional.transport(PsiPi)
rot_suite = AxiomSuite(rotated, registry=reg, probe=h, structures=(N3, N6))
print(rot_suite.run(x1, x2, f), f"\n({time.time()-t:.1f}s)\n")
print("engine assembled for the rotated checks:")
print("\n".join(rot_suite.last_report))

The two rules the first notebook had to write by hand — graded
commutativity of the wedge inside operator slots and scalar pull-out from a
wedge factor — appear in that report as library rules
(`WedgeGradedOrderDefinition`, `WedgeScalarFactorDefinition`), added because
the scan found wedges.

### 2.3 Term separation, still available

The decomposition theorems of the first notebook (rotated form slots =
original + Π-block through `ℒ` and `ι`; vector slot = shifted Lie bracket −
Π of the rotated forms) are library theorems and run as before; the rotated
bracket built here through the matrix is the same expression up to
normalisation.

In [ ]:
from jacopy.packages.generalized.exceptional_rotation import (
    rotated_exceptional_bracket, prove_rotated_form_decomposition, prove_rotated_vector_decomposition,
)
lib = T3.section(*rotated_exceptional_bracket(N3, N6, U, om2, om5, V, et2, et5))
mine = rotated.bracket(x1, x2)
eng = assemble_engine(*lib, *mine, registry=reg, structures=(N3, N6))
print("matrix-built − library-built, normalised:", [str(_normalized_by(eng, Sum(a, Neg(b)), reg)) for a, b in zip(mine, lib)])
for label, prover in (("form slots: original + Π-block through ℒ and ι", prove_rotated_form_decomposition),
                      ("vector slot: shifted Lie bracket − Π(rotated forms)", prove_rotated_vector_decomposition)):
    chain, thm = prover(N3, N6, U, om2, om5, V, et2, et5, registry=reg)
    print(f"CLOSED  {label} ({len(chain.steps)} steps)")

### 2.4 The frame formula (4.13)

The paper writes `Π₃ ⊛ Π₃` in a frame as `5 Π^{[a₁a₂a₃} Π^{a₄a₅]c}`, with the
bracket meaning antisymmetrisation over the five indices. The antisymmetriser
builds exactly that expression from the component product and we verify the
two properties that make it a `[…]`: swapping any two of the bracketed
indices flips the sign, while the free index `c` is untouched.

In [ ]:
from jacopy.core.expr import Product
from jacopy.core.multi_eval import MultiEval
from jacopy.central.objects import frame
from jacopy.algorithms.simplify import simplify

fr = frame(); co = fr.dual()
def Pi3(*idx):                                          # the component Π₃^{abc} = Π₃(e^a, e^b, e^c)
    return MultiEval(N3.pi, *(co.field(i) for i in idx), alternating=True, slot_kind="covector")

product = Product(Pi3("a1", "a2", "a3"), Pi3("a4", "a5", "c"))
anti = antisymmetrize(product, ("a1", "a2", "a3", "a4", "a5"))               # Π^{[a1a2a3} Π^{a4a5]c}  (with the 1/5!)
formula_413 = Product(Integer(5), anti)                                      # 5 Π^{[a1a2a3} Π^{a4a5]c}
signed_sum = anti.children[1]
print("number of signed terms in the antisymmetrisation:", len(signed_sum.children))
print("first term:", signed_sum.children[0]._repr_inner())
for a, b in (("a1", "a2"), ("a3", "a4"), ("a1", "a5")):
    print(f"formula + formula with {a}↔{b} simplifies to:", simplify(Sum(formula_413, swap_indices(formula_413, a, b)), reg))
print("the free index c stays free:", "c" in formula_413._repr_inner())

Whether this antisymmetrised product *is* the `⊛` map of §2.1 was the one
question the first version of this notebook could not answer: the map is
`½ Π₃(ι_{Π₃} ω₅)`, and `ι_{Π₃}` had no rule for evaluating on a basis
5-form. It has one now — the component face of the partial contraction,
`ι_P(α₁∧…∧α_q) = Σ_I sgn(I,Iᶜ) P(α_I) α_{Iᶜ}`, whose sign convention is
forced by the decomposable-multivector rule and checked against it on every
`(p, q)` in the test suite. With it, `⟨e^c, ⊛(e^{a₁}∧…∧e^{a₅})⟩` normalises
to ten `½`-weighted products of components, and so does the paper's
`5 Π^{[a₁a₂a₃} Π^{a₄a₅]c}` (each split of the five indices into 3 + 2
appears `3!·2! = 12` times in the `5!` permutations, and `5·12/5! = ½`).

In [ ]:
from jacopy.packages.generalized.frame_evaluation import prove_boxtimes_components_are_413

t = time.time()
chain, thm = prove_boxtimes_components_are_413(N3, fr, ("a1", "a2", "a3", "a4", "a5"), "c", registry=reg)
for s in chain.steps:
    print(f"CLOSED  {s.rule}")
print("canonical form of ⟨e^c, ⊛(e^{a1…a5})⟩:", chain.steps[0].after._repr_inner()[:150], "…")
print(f"({time.time()-t:.2f}s)")

### 2.5 The 5-form slot of the Jacobi identity — closed by linearity

The one condition the first notebook left open was the Leibniz–Jacobi
identity on the 5-form slot, where the cross-term enters a third-order
computation. Brute force — three nested brackets evaluated on five vector
slots — does not finish (it is not a cycle, just combinatorial size: the
2-form slot alone expands to about 5000 nodes before collapsing to 0).

The way through is linearity. The bracket's 5-form slot is
`L(x, y₅) + C(x₂, y₂)` — the Dorfman formula plus the cross-term — so the
Jacobiator splits *exactly* into a Dorfman Jacobiator on the 5-forms plus a
cross Jacobiator built from the 2-forms only. The split identity and the
cross part both close at **form level** (no slot evaluation, well under a
second); the Dorfman part is the library's degree-general theorem, proven on
five slots in about 30 s (the slow test suite) — here it is cited.

Nothing in this is special to "five": the prover is `prove_top_slot_jacobi`
for the whole family `TM ⊕ Λᵖ ⊕ Λ^{2p+1}` (`p = 2` is the E₆ case, and
`prove_exceptional_jacobi_five` is just that instance). Asking the question
turned up a fact, though: the single cross-term bracket is a Leibniz bracket
for **even** `p` only — for odd `p` the cross Jacobiator is
`∓2 dη_p ∧ ι_W dω_p`, for every sign convention, and an independent
brute-force expansion at `p = 1` confirms the non-zero residual. The prover
fails honestly there and shows it.

In [ ]:
from jacopy.packages.drinfeld.examples import prove_exceptional_jacobi_five

X5 = vector_fields("X₁ X₂ X₃ X₄ X₅")
t = time.time()
chain, assumptions = prove_exceptional_jacobi_five(N3, U, om2, om5, V, et2, et5, W, ze2, ze5, X5,
                                                   registry=reg, cite_dorfman=True)
for s in chain.steps:
    print(f"CLOSED  {s.rule}")
print("assumptions:", assumptions, f"({time.time()-t:.2f}s)")

# the same prover across the family: even p closes, odd p is obstructed
from jacopy.packages.drinfeld.examples import prove_top_slot_jacobi
from jacopy.proof.strategies import ProofFailure
for p in (1, 2, 3, 4):
    a, b, c = forms("α β γ", degree=p); A, B, C = forms("A B C", degree=2 * p + 1)
    slots = vector_fields(" ".join(f"Y{i}" for i in range(2 * p + 1)))
    try:
        prove_top_slot_jacobi(nambu_structure("Π", p=p), U, a, A, V, b, B, W, c, C, slots, registry=reg, cite_dorfman=True)
        print(f"p = {p} (TM ⊕ Λ{p} ⊕ Λ{2*p+1}): CLOSED")
    except ProofFailure as exc:
        print(f"p = {p} (TM ⊕ Λ{p} ⊕ Λ{2*p+1}): OBSTRUCTED —", str(exc).split("residual ")[1][:60])

### 2.6 Identification with the exceptional Drinfel'd double — the mixed `R′` conditions

Part 1 ended with the fact that the `Ψ_θ`-twisted Dorfman bracket *is* the
Nambu double up to the derived twist `R′(ω,η) = [θ♯ω, θ♯η] − θ♯[ω,η]_θ`, and
that `R′ = 0` is the Poisson condition. The same question for the rotation
of Part 2 has the same answer, with the bivector replaced by the whole block
map of the matrix `Ψ_Π`,

`Π(ω₂, ω₅) = Π₃ω₂ + Π̂ω₅`, `Π̂ := Π₆ + Π₃⊛Π₃`,

acting from `Z = Λ² ⊕ Λ⁵` to `TM`. Write the double of `TM` and `Z` with
the Nambu double's formula and this `Π` (tilde Lie derivative
`ℒ̃_ω W = [Πω, W] + Π(ι_W dω)`, `d̃ = −Πd`, and the Z-bracket
`[ω,η]_Z = ℒ_{Πω}η − ι_{Πη}dω` on both slots plus the cross-term `−η₂∧dω₂`
in the 5-slot). Then, with no assumption on `Π₃, Π₆`,

`Ψ_Π⁻¹[Ψ_Π x, Ψ_Π y] = [x, y]_double + (R′(ω,η), 0, 0)`,  `R′(ω,η) = [Πω,Πη] − Π[ω,η]_Z`.

`R′` is the anchor defect of the Z-bracket under the block map, and it is
bilinear in `(ω₂,ω₅)` and `(η₂,η₅)`, so it splits *exactly* into four
bidegree pieces — these are the conditions for the rotated bracket to be an
exceptional Drinfel'd double, and they mix `Π₃`, `Π₆` and `⊛`:

| piece | closed form | reading |
| --- | --- | --- |
| `R′₂₂(ω₂,η₂)` | `[Π₃ω₂,Π₃η₂] − Π₃[ω₂,η₂]_{Π₃} + Π̂(η₂∧dω₂)` | Nambu derived twist of `Π₃` **plus** the cross-term lifted by `Π̂` |
| `R′₂₅(ω₂,η₅)` | `[Π₃ω₂,Π̂η₅] − Π̂(ℒ_{Π₃ω₂}η₅) + Π₃(ι_{Π̂η₅}dω₂)` | `= ℒ̃_{ω₂}(Π̂η₅) − Π̂(ℒ_{Π₃ω₂}η₅)`: `Π̂` is `ℒ̃`-equivariant |
| `R′₅₂(ω₅,η₂)` | `[Π̂ω₅,Π₃η₂] − Π₃(ℒ_{Π̂ω₅}η₂) + Π̂(ι_{Π₃η₂}dω₅)` | the mirror mixed condition |
| `R′₅₅(ω₅,η₅)` | `[Π̂ω₅,Π̂η₅] − Π̂(ℒ_{Π̂ω₅}η₅ − ι_{Π̂η₅}dω₅)` | the anchor defect of the map `Π̂ : Λ⁵ → TM` for its own Koszul-type bracket (`Π̂` is *not* a 6-vector in general: `Π₃⊛Π₃` is alternating in five slots only) |

Two more facts come for free. If `Π₃` is declared Nambu–Poisson, its own
derived twist dies and the first condition becomes `Π̂(η₂∧dω₂) = 0` for all
2-forms — the exceptional structure ties the `Λ⁵ → TM` block to the
trivector through the M-theory term; without the declaration the prover
shows the residual. And the symmetric part of `R′` is the block map on the
*exact* transported pairing, `R′(ω,η) + R′(η,ω) = −Π₃dP₂′ − Π̂dP₅′` with
`P′ = ⟨Ψ_Πω, Ψ_Πη⟩`, so the Drinfel'd conditions force `Π D′⟨ω,η⟩ = 0` —
the exceptional face of the `g_Z` consequence of the fundamental identity
seen in Part 1.

In [ ]:
from jacopy.packages.generalized.exceptional_double import (
    exceptional_double, exceptional_r_twist, exceptional_r_twist_components,
    prove_rotated_is_exceptional_double_plus_r, prove_exceptional_r_twist_bidegree_split,
    prove_r22_reduces_to_lifted_cross_term, prove_r25_is_lie_tilde_equivariance_defect,
    prove_exceptional_r_twist_symmetric_part,
)
from jacopy.proof.strategies import ProofFailure

t = time.time()
# the rotated bracket built by the matrix, minus the double, minus (R′, 0, 0) — through the interface
double = T3.section(*exceptional_double(N3, N6, U, om2, om5, V, et2, et5))
Rp = T3.section(exceptional_r_twist(N3, N6, om2, om5, et2, et5), Integer(0), Integer(0))
diff = rotated.bracket(x1, x2) - double - Rp
eng = assemble_engine(*diff, registry=reg, structures=(N3, N6))
print("matrix-rotated − double − (R′,0,0), normalised:",
      [str(_normalized_by(eng, Act(diff[0], h), reg))] + [str(_normalized_by(eng, c, reg)) for c in diff[1:]])

for label, prover, args in (
    ("rotated = double + (R′,0,0)", prove_rotated_is_exceptional_double_plus_r, (U, om2, om5, V, et2, et5, h)),
    ("R′ = R′₂₂ + R′₂₅ + R′₅₂ + R′₅₅", prove_exceptional_r_twist_bidegree_split, (om2, om5, et2, et5, h)),
    ("R′₂₅ = ℒ̃-equivariance defect of Π̂", prove_r25_is_lie_tilde_equivariance_defect, (om2, et5, h)),
    ("sym R′ = −Π₃dP₂′ − Π̂dP₅′", prove_exceptional_r_twist_symmetric_part, (om2, om5, et2, et5, h)),
):
    chain, thm = prover(N3, N6, *args, registry=reg)
    print(f"CLOSED  {label}  ({len(chain.steps)} step(s); assumptions: {thm.from_axioms[-1]})")

chain, thm = prove_r22_reduces_to_lifted_cross_term(N3, N6, om2, et2, h, registry=reg, declare_fi=True)
print("CLOSED  R′₂₂ = Π̂(η₂∧dω₂)  under the declared fundamental identity of Π₃")
try:
    prove_r22_reduces_to_lifted_cross_term(N3, N6, om2, et2, h, registry=reg, declare_fi=False)
except ProofFailure as exc:
    print("HONEST  without the declaration:", str(exc)[:110], "…")
print(f"({time.time()-t:.1f}s)")

## What changed between the two notebooks — and what is still open

Same mathematics, same honesty, roughly a third of the code: matrices are
matrices, sections are sections, the engine is built and explained
automatically, and one method (`transport`) is the paper's eq. (7.3) for a
whole structure. Three facts became visible that the first notebook could
not show cheaply:

* the **Leibniz–Jacobi identity** of the exceptional bracket closes on all
  three slots — vector and 2-form through the suite's bracket-identity
  repair loop, the 5-form slot through the linear split of §2.5;
* the **anchor-morphism** condition of the rotated bracket closes — for the
  transported anchor `ρ∘Ψ_Π` it is the transport of the trivial one, so no
  condition on `Π₃, Π₆` appears here. The exceptional analogue of the Poisson
  condition lives, as in Part 1, in the *identification* of the rotated
  bracket with the exceptional Drinfel'd double — §2.6: the four bidegree
  pieces of the derived twist `R′`, mixing `Π₃`, `Π₆` and `⊛`, with the
  cross-term lifted by `Π₆ + Π₃⊛Π₃` as the genuinely exceptional condition;
* the twisted Dorfman bracket satisfies Jacobi for **any** bivector (about a
  minute per component; in the slow test suite);
* the **metric-invariance** condition with the R-valued action closes for
  the exceptional bracket (§2.2) and, with a large expansion budget, for
  its rotation too — the 4-form component takes about 13 minutes, so that
  check lives in the very-slow test suite rather than here.

Nothing from the original question is left open. The frame formula (4.13)
is identified with the `⊛` map component-wise (§2.4), the higher-order
Nambu conditions run at `p = 2, 3` (§1.5), and the two dialects of the
order-1 objects are bridged (§1.6). What remains is outside the question:
the general-`p` analogue of the exceptional bundle `TM ⊕ Λᵖ ⊕ Λ^{2p+1}` is
a Leibniz bracket only for even `p` (§2.5), and the rotated metric
invariance is a matter of expansion budget, not of rules.